# CS432 — Assignment 3: ACID Transaction Validation
## Module A: Transaction Management & Crash Recovery
### StayEase Hostel Management System — B+ Tree Database Engine

---
## Setup: Initialize Database

In [1]:
import sys
import os
import json
import threading
import time

sys.path.insert(0, os.path.abspath('.'))

from db_manager import DatabaseManager

db_admin = DatabaseManager()
db_admin.create_database("stayease")

members_schema = {
    'Member_ID': int, 'First_Name': str, 'Last_Name': str,
    'Gender': str, 'Age': int, 'Contact_Number': str,
    'Email': str, 'Emergency_Contact': str, 'Image_Path': str,
    'Role_ID': int, 'Status': str
}
rooms_schema = {
    'Room_ID': int, 'Room_Number': str, 'Hostel_ID': int,
    'Floor_Number': int, 'Capacity': int
}
allocations_schema = {
    'Allocation_ID': int, 'Member_ID': int, 'Room_ID': int,
    'Allocation_Date': str, 'Check_Out_Date': str, 'Status': str
}

db_admin.create_table("stayease", "members", members_schema, search_key="Member_ID")
db_admin.create_table("stayease", "rooms", rooms_schema, search_key="Room_ID")
db_admin.create_table("stayease", "room_allocations", allocations_schema, search_key="Allocation_ID")

db_admin.reload_all("stayease")

members_table, _ = db_admin.get_table("stayease", "members")
rooms_table, _   = db_admin.get_table("stayease", "rooms")
alloc_table, _   = db_admin.get_table("stayease", "room_allocations")

if not members_table.get(110)[0] or not rooms_table.get(101)[0] or not members_table.get(112)[0] or not alloc_table.get(1)[0] or not alloc_table.get(2)[0]:
    db_admin.begin()
    if not members_table.get(110)[0]:
        members_table.insert({'Member_ID': 110, 'First_Name': 'Priya', 'Last_Name': 'Sharma',
            'Gender': 'F', 'Age': 20, 'Contact_Number': '8888800001',
            'Email': 'priya@student.com', 'Status': 'Active',
            'Role_ID': 1, 'Image_Path': 'uploads/default.jpg', 'Emergency_Contact': '7777700001'})
    if not members_table.get(112)[0]:
        members_table.insert({'Member_ID': 112, 'First_Name': 'Ravi', 'Last_Name': 'Kumar',
            'Gender': 'M', 'Age': 22, 'Contact_Number': '8888800002',
            'Email': 'ravi@student.com', 'Status': 'Active',
            'Role_ID': 1, 'Image_Path': 'uploads/default.jpg', 'Emergency_Contact': '7777700002'})
    if not rooms_table.get(101)[0]:
        rooms_table.insert({'Room_ID': 101, 'Room_Number': '101', 'Hostel_ID': 2,
            'Floor_Number': 1, 'Capacity': 2})
    if not alloc_table.get(1)[0]:
        alloc_table.insert({'Allocation_ID': 1, 'Member_ID': 110, 'Room_ID': 101,
            'Allocation_Date': '2026-01-01', 'Check_Out_Date': 'None', 'Status': 'Active'})
    if not alloc_table.get(2)[0]:
        alloc_table.insert({'Allocation_ID': 2, 'Member_ID': 112, 'Room_ID': 101,
            'Allocation_Date': '2026-01-01', 'Check_Out_Date': 'None', 'Status': 'Active'})
    db_admin.commit("stayease")

db_admin.persist_all("stayease")
print("\nDatabase initialized and ready.")

Restored table 'members' from disk.
Restored table 'rooms' from disk.
Restored table 'room_allocations' from disk.
All tables in 'stayease' persisted to disk.

Database initialized and ready.


---
## Helper Functions

In [2]:
def print_table(table_obj, title):
    records = table_obj.get_all()
    print(f"\n{'='*60}")
    print(f"  TABLE: {title}  ({len(records)} records)")
    print(f"{'='*60}")
    if not records:
        print("  (empty)")
    for key, record in records:
        print(f"  {record}")
    print(f"{'='*60}")

def print_all_tables():
    m, _ = db_admin.get_table("stayease", "members")
    r, _ = db_admin.get_table("stayease", "rooms")
    a, _ = db_admin.get_table("stayease", "room_allocations")
    print_table(m, "Members")
    print_table(r, "Rooms")
    print_table(a, "Room Allocations")

def print_wal_log():
    log_file = db_admin.log_file
    print(f"\n{'='*60}")
    print(f"  WAL LOG: {log_file}")
    print(f"{'='*60}")
    if not os.path.exists(log_file):
        print("  (log file does not exist)")
        print(f"{'='*60}")
        return
    with open(log_file, 'r') as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]
    if not lines:
        print("  (log is empty)")
    for line in lines:
        entry = json.loads(line)
        op = entry['op']
        tx_id = entry.get('tx_id', 'N/A')
        ts = entry.get('timestamp', 'N/A')
        if op in ('BEGIN', 'COMMIT'):
            print(f"  >>> {op:8} | tx_id={tx_id} | time={ts}")
        else:
            print(f"  {op:8} | tx_id={tx_id} | time={ts} | table={entry['table']} | key={entry['key']}")
    print(f"{'='*60}")

def section(title):
    print(f"\n{'#'*60}")
    print(f"  {title}")
    print(f"{'#'*60}")

print("Helper functions ready.")

Helper functions ready.


---
## Current State of Database

In [3]:
section("CURRENT DATABASE STATE")
print_all_tables()
print_wal_log()


############################################################
  CURRENT DATABASE STATE
############################################################

  TABLE: Members  (2 records)
  {'Member_ID': 110, 'First_Name': 'Priya', 'Last_Name': 'Sharma', 'Gender': 'F', 'Age': 20, 'Contact_Number': '8888800001', 'Email': 'priya@student.com', 'Status': 'Active', 'Role_ID': 1, 'Image_Path': 'uploads/default.jpg', 'Emergency_Contact': '7777700001'}
  {'Member_ID': 112, 'First_Name': 'Ravi', 'Last_Name': 'Kumar', 'Gender': 'M', 'Age': 22, 'Contact_Number': '8888800002', 'Email': 'ravi@student.com', 'Status': 'Active', 'Role_ID': 1, 'Image_Path': 'uploads/default.jpg', 'Emergency_Contact': '7777700002'}

  TABLE: Rooms  (1 records)
  {'Room_ID': 101, 'Room_Number': '101', 'Hostel_ID': 2, 'Floor_Number': 1, 'Capacity': 2}

  TABLE: Room Allocations  (2 records)
  {'Allocation_ID': 1, 'Member_ID': 110, 'Room_ID': 101, 'Allocation_Date': '2026-01-01', 'Check_Out_Date': 'None', 'Status': 'Active'}
  {'Al

---
## TEST 1: ATOMICITY
### If any step in a multi-table transaction fails, ALL changes must be rolled back.

In [4]:
section("TEST 1: ATOMICITY — Multi-Table Rollback on Failure")

members_table, _ = db_admin.get_table("stayease", "members")
rooms_table, _   = db_admin.get_table("stayease", "rooms")
alloc_table, _   = db_admin.get_table("stayease", "room_allocations")

_, member_before = members_table.get(112)
_, room_before   = rooms_table.get(101)
alloc_ids_before = [k for k, v in alloc_table.get_all()]

print("\n--- STATE BEFORE TRANSACTION ---")
print(f"  Member 112 Status : {member_before['Status']}")
print(f"  Room 101 Capacity : {room_before['Capacity']}")
print(f"  Allocation IDs    : {alloc_ids_before}")
print_all_tables()
print_wal_log()


############################################################
  TEST 1: ATOMICITY — Multi-Table Rollback on Failure
############################################################

--- STATE BEFORE TRANSACTION ---
  Member 112 Status : Active
  Room 101 Capacity : 2
  Allocation IDs    : [1, 2]

  TABLE: Members  (2 records)
  {'Member_ID': 110, 'First_Name': 'Priya', 'Last_Name': 'Sharma', 'Gender': 'F', 'Age': 20, 'Contact_Number': '8888800001', 'Email': 'priya@student.com', 'Status': 'Active', 'Role_ID': 1, 'Image_Path': 'uploads/default.jpg', 'Emergency_Contact': '7777700001'}
  {'Member_ID': 112, 'First_Name': 'Ravi', 'Last_Name': 'Kumar', 'Gender': 'M', 'Age': 22, 'Contact_Number': '8888800002', 'Email': 'ravi@student.com', 'Status': 'Active', 'Role_ID': 1, 'Image_Path': 'uploads/default.jpg', 'Emergency_Contact': '7777700002'}

  TABLE: Rooms  (1 records)
  {'Room_ID': 101, 'Room_Number': '101', 'Hostel_ID': 2, 'Floor_Number': 1, 'Capacity': 2}

  TABLE: Room Allocations  (2 record

In [5]:
print("\n--- STARTING TRANSACTION (will fail at Step 3) ---")
db_admin.begin()

try:
    updated_member = member_before.copy()
    updated_member['Status'] = 'Inactive'
    members_table.update(112, updated_member)
    print("  [Step 1] Member 112 status set to Inactive.")

    updated_room = room_before.copy()
    updated_room['Capacity'] = 99
    rooms_table.update(101, updated_room)
    print("  [Step 2] Room 101 capacity set to 99.")

    print("  [Step 3] Attempting invalid allocation insert...")
    success, msg = alloc_table.insert({'Allocation_ID': 999})
    if not success:
        raise ValueError(f"Simulated Failure: {msg}")

    db_admin.commit("stayease")

except Exception as e:
    print(f"\n  ERROR: {e}")
    print("  Triggering ROLLBACK...")
    db_admin.rollback()

print("\n--- WAL LOG AFTER ROLLBACK (incomplete tx removed) ---")
print_wal_log()


--- STARTING TRANSACTION (will fail at Step 3) ---
BEGIN: Transaction started. [tx_id=b07dab61]
  [Step 1] Member 112 status set to Inactive.
  [Step 2] Room 101 capacity set to 99.
  [Step 3] Attempting invalid allocation insert...

  ERROR: Simulated Failure: Missing column 'Member_ID' in record.
  Triggering ROLLBACK...
ROLLBACK: All changes reverted. Database is back to original state.

--- WAL LOG AFTER ROLLBACK (incomplete tx removed) ---

  WAL LOG: stayease_wal.log
  >>> BEGIN    | tx_id=24fd21b7 | time=2026-04-04 22:25:45
  INSERT   | tx_id=24fd21b7 | time=2026-04-04 22:25:03 | table=members | key=110
  INSERT   | tx_id=24fd21b7 | time=2026-04-04 22:25:03 | table=members | key=112
  INSERT   | tx_id=24fd21b7 | time=2026-04-04 22:25:03 | table=rooms | key=101
  INSERT   | tx_id=24fd21b7 | time=2026-04-04 22:25:03 | table=room_allocations | key=1
  INSERT   | tx_id=24fd21b7 | time=2026-04-04 22:25:03 | table=room_allocations | key=2
  >>> COMMIT   | tx_id=24fd21b7 | time=2026-0

In [6]:
print("\n--- STATE AFTER ROLLBACK ---")
_, member_after = members_table.get(112)
_, room_after   = rooms_table.get(101)
alloc_ids_after = [k for k, v in alloc_table.get_all()]

print(f"  Member 112 Status : {member_after['Status']}  (Expected: Active)")
print(f"  Room 101 Capacity : {room_after['Capacity']}  (Expected: 2)")
print(f"  Allocation IDs    : {alloc_ids_after}  (Expected: [1, 2] — no 999)")
print_all_tables()

atomicity_ok = (
    member_after['Status'] == 'Active' and
    room_after['Capacity'] == 2 and
    999 not in alloc_ids_after
)
print("\n" + ("ATOMICITY TEST PASSED" if atomicity_ok else "ATOMICITY TEST FAILED"))


--- STATE AFTER ROLLBACK ---
  Member 112 Status : Active  (Expected: Active)
  Room 101 Capacity : 2  (Expected: 2)
  Allocation IDs    : [1, 2]  (Expected: [1, 2] — no 999)

  TABLE: Members  (2 records)
  {'Member_ID': 110, 'First_Name': 'Priya', 'Last_Name': 'Sharma', 'Gender': 'F', 'Age': 20, 'Contact_Number': '8888800001', 'Email': 'priya@student.com', 'Status': 'Active', 'Role_ID': 1, 'Image_Path': 'uploads/default.jpg', 'Emergency_Contact': '7777700001'}
  {'Member_ID': 112, 'First_Name': 'Ravi', 'Last_Name': 'Kumar', 'Gender': 'M', 'Age': 22, 'Contact_Number': '8888800002', 'Email': 'ravi@student.com', 'Status': 'Active', 'Role_ID': 1, 'Image_Path': 'uploads/default.jpg', 'Emergency_Contact': '7777700002'}

  TABLE: Rooms  (1 records)
  {'Room_ID': 101, 'Room_Number': '101', 'Hostel_ID': 2, 'Floor_Number': 1, 'Capacity': 2}

  TABLE: Room Allocations  (2 records)
  {'Allocation_ID': 1, 'Member_ID': 110, 'Room_ID': 101, 'Allocation_Date': '2026-01-01', 'Check_Out_Date': 'None'

---
## TEST 2: CONSISTENCY
### After every transaction, all relations must remain valid — no orphan records.

In [7]:
section("TEST 2: CONSISTENCY — Relations Stay Valid After Operations")

members_table, _ = db_admin.get_table("stayease", "members")
rooms_table, _   = db_admin.get_table("stayease", "rooms")
alloc_table, _   = db_admin.get_table("stayease", "room_allocations")

print("\n--- STATE BEFORE CONSISTENCY TEST ---")
print_all_tables()
print_wal_log()

print("\n--- SCENARIO 1: Successful commit — all 3 tables updated together ---")
db_admin.begin()
try:
    members_table.insert({'Member_ID': 130, 'First_Name': 'Test', 'Last_Name': 'User',
        'Gender': 'M', 'Age': 21, 'Contact_Number': '9000000001',
        'Email': 'testuser@student.com', 'Status': 'Active',
        'Role_ID': 1, 'Image_Path': 'uploads/default.png', 'Emergency_Contact': '9000000002'})
    rooms_table.insert({'Room_ID': 303, 'Room_Number': '303', 'Hostel_ID': 2,
        'Floor_Number': 3, 'Capacity': 2})
    alloc_table.insert({'Allocation_ID': 20, 'Member_ID': 130, 'Room_ID': 303,
        'Allocation_Date': '2026-04-01', 'Check_Out_Date': 'None', 'Status': 'Active'})
    db_admin.commit("stayease")
    print("  Transaction committed successfully.")
except Exception as e:
    print(f"  ERROR: {e}")
    db_admin.rollback()

print("\n--- WAL LOG AFTER COMMIT ---")
print_wal_log()


############################################################
  TEST 2: CONSISTENCY — Relations Stay Valid After Operations
############################################################

--- STATE BEFORE CONSISTENCY TEST ---

  TABLE: Members  (2 records)
  {'Member_ID': 110, 'First_Name': 'Priya', 'Last_Name': 'Sharma', 'Gender': 'F', 'Age': 20, 'Contact_Number': '8888800001', 'Email': 'priya@student.com', 'Status': 'Active', 'Role_ID': 1, 'Image_Path': 'uploads/default.jpg', 'Emergency_Contact': '7777700001'}
  {'Member_ID': 112, 'First_Name': 'Ravi', 'Last_Name': 'Kumar', 'Gender': 'M', 'Age': 22, 'Contact_Number': '8888800002', 'Email': 'ravi@student.com', 'Status': 'Active', 'Role_ID': 1, 'Image_Path': 'uploads/default.jpg', 'Emergency_Contact': '7777700002'}

  TABLE: Rooms  (1 records)
  {'Room_ID': 101, 'Room_Number': '101', 'Hostel_ID': 2, 'Floor_Number': 1, 'Capacity': 2}

  TABLE: Room Allocations  (2 records)
  {'Allocation_ID': 1, 'Member_ID': 110, 'Room_ID': 101, 'Allocati

In [8]:
member_exists, _          = members_table.get(130)
room_exists, _            = rooms_table.get(303)
alloc_exists, alloc_data  = alloc_table.get(20)

ref_member_ok, ref_room_ok = False, False
if alloc_exists:
    ref_member_ok, _ = members_table.get(alloc_data['Member_ID'])
    ref_room_ok, _   = rooms_table.get(alloc_data['Room_ID'])

print("\n--- REFERENTIAL INTEGRITY CHECK ---")
print(f"  Member 130 exists                         : {member_exists}  (Expected: True)")
print(f"  Room 303 exists                           : {room_exists}  (Expected: True)")
print(f"  Allocation 20 exists                      : {alloc_exists}  (Expected: True)")
print(f"  Allocation Member_ID refs valid member    : {ref_member_ok}  (Expected: True)")
print(f"  Allocation Room_ID refs valid room        : {ref_room_ok}  (Expected: True)")
print_all_tables()


--- REFERENTIAL INTEGRITY CHECK ---
  Member 130 exists                         : True  (Expected: True)
  Room 303 exists                           : True  (Expected: True)
  Allocation 20 exists                      : True  (Expected: True)
  Allocation Member_ID refs valid member    : True  (Expected: True)
  Allocation Room_ID refs valid room        : True  (Expected: True)

  TABLE: Members  (3 records)
  {'Member_ID': 110, 'First_Name': 'Priya', 'Last_Name': 'Sharma', 'Gender': 'F', 'Age': 20, 'Contact_Number': '8888800001', 'Email': 'priya@student.com', 'Status': 'Active', 'Role_ID': 1, 'Image_Path': 'uploads/default.jpg', 'Emergency_Contact': '7777700001'}
  {'Member_ID': 112, 'First_Name': 'Ravi', 'Last_Name': 'Kumar', 'Gender': 'M', 'Age': 22, 'Contact_Number': '8888800002', 'Email': 'ravi@student.com', 'Status': 'Active', 'Role_ID': 1, 'Image_Path': 'uploads/default.jpg', 'Emergency_Contact': '7777700002'}
  {'Member_ID': 130, 'First_Name': 'Test', 'Last_Name': 'User', 'Gen

In [9]:
print("\n--- SCENARIO 2: Partial insert must not persist after rollback ---")
db_admin.begin()
try:
    members_table.insert({'Member_ID': 131, 'First_Name': 'Ghost', 'Last_Name': 'User',
        'Gender': 'M', 'Age': 22, 'Contact_Number': '9000000003',
        'Email': 'ghost@student.com', 'Status': 'Active',
        'Role_ID': 1, 'Image_Path': 'uploads/default.png', 'Emergency_Contact': '9000000004'})
    print("  Inserted Ghost Member 131 (partial).")
    raise RuntimeError("Simulated crash before completing transaction.")
    db_admin.commit("stayease")
except Exception as e:
    print(f"  ERROR: {e}")
    db_admin.rollback()

ghost_exists, _ = members_table.get(131)
print(f"\n  Ghost Member 131 exists: {ghost_exists}  (Expected: False)")

print("\n--- WAL LOG AFTER ROLLBACK (ghost tx removed) ---")
print_wal_log()

consistency_ok = member_exists and room_exists and alloc_exists and ref_member_ok and ref_room_ok and not ghost_exists
print("\n" + ("CONSISTENCY TEST PASSED" if consistency_ok else "CONSISTENCY TEST FAILED"))

print("\n--- CLEANUP ---")
db_admin.begin()
alloc_table.delete(20)
rooms_table.delete(303)
members_table.delete(130)
db_admin.commit("stayease")
print("  Test records removed.")
print_all_tables()


--- SCENARIO 2: Partial insert must not persist after rollback ---
BEGIN: Transaction started. [tx_id=4f3d4c75]
  Inserted Ghost Member 131 (partial).
  ERROR: Simulated crash before completing transaction.
ROLLBACK: All changes reverted. Database is back to original state.

  Ghost Member 131 exists: False  (Expected: False)

--- WAL LOG AFTER ROLLBACK (ghost tx removed) ---

  WAL LOG: stayease_wal.log
  >>> BEGIN    | tx_id=24fd21b7 | time=2026-04-04 22:25:45
  INSERT   | tx_id=24fd21b7 | time=2026-04-04 22:25:03 | table=members | key=110
  INSERT   | tx_id=24fd21b7 | time=2026-04-04 22:25:03 | table=members | key=112
  INSERT   | tx_id=24fd21b7 | time=2026-04-04 22:25:03 | table=rooms | key=101
  INSERT   | tx_id=24fd21b7 | time=2026-04-04 22:25:03 | table=room_allocations | key=1
  INSERT   | tx_id=24fd21b7 | time=2026-04-04 22:25:03 | table=room_allocations | key=2
  >>> COMMIT   | tx_id=24fd21b7 | time=2026-04-04 22:25:45
  >>> BEGIN    | tx_id=9a2e4367 | time=2026-04-04 22:25:

---
## TEST 3: ISOLATION
### Concurrent transactions must be serialized. No dirty reads or corruption.

In [10]:
section("TEST 3: ISOLATION — Concurrent Transactions Serialized by Lock")

results = {}

def transaction_thread_1():
    print("  [Thread 1] Waiting to acquire lock...")
    db_admin.begin()
    try:
        rooms_table, _ = db_admin.get_table("stayease", "rooms")
        _, room = rooms_table.get(101)
        print(f"  [Thread 1] Room 101 Capacity BEFORE: {room['Capacity']}")
        updated = room.copy()
        updated['Capacity'] = 5
        rooms_table.update(101, updated)
        print("  [Thread 1] Set capacity to 5. Sleeping 2 seconds (holding lock)...")
        time.sleep(2)
        restored = room.copy()
        restored['Capacity'] = 2
        rooms_table.update(101, restored)
        db_admin.commit("stayease")
        print("  [Thread 1] Committed. Lock released.")
        results['thread1'] = 'committed'
    except Exception as e:
        print(f"  [Thread 1] Error: {e}")
        db_admin.rollback()
        results['thread1'] = 'rolled_back'

def transaction_thread_2():
    time.sleep(0.3)
    print("  [Thread 2] Waiting to acquire lock...")
    db_admin.begin()
    try:
        members_table, _ = db_admin.get_table("stayease", "members")
        _, member = members_table.get(112)
        print(f"  [Thread 2] Member 112 Status BEFORE: {member['Status']}")
        updated = member.copy()
        updated['Status'] = 'Inactive'
        members_table.update(112, updated)
        restored = member.copy()
        restored['Status'] = 'Active'
        members_table.update(112, restored)
        db_admin.commit("stayease")
        print("  [Thread 2] Committed.")
        results['thread2'] = 'committed'
    except Exception as e:
        print(f"  [Thread 2] Error: {e}")
        db_admin.rollback()
        results['thread2'] = 'rolled_back'

print("\n--- STATE BEFORE ISOLATION TEST ---")
print_all_tables()

print("\n--- LAUNCHING TWO CONCURRENT THREADS ---")
t1 = threading.Thread(target=transaction_thread_1, name="Thread-1")
t2 = threading.Thread(target=transaction_thread_2, name="Thread-2")
t1.start()
t2.start()
t1.join()
t2.join()


############################################################
  TEST 3: ISOLATION — Concurrent Transactions Serialized by Lock
############################################################

--- STATE BEFORE ISOLATION TEST ---

  TABLE: Members  (2 records)
  {'Member_ID': 110, 'First_Name': 'Priya', 'Last_Name': 'Sharma', 'Gender': 'F', 'Age': 20, 'Contact_Number': '8888800001', 'Email': 'priya@student.com', 'Status': 'Active', 'Role_ID': 1, 'Image_Path': 'uploads/default.jpg', 'Emergency_Contact': '7777700001'}
  {'Member_ID': 112, 'First_Name': 'Ravi', 'Last_Name': 'Kumar', 'Gender': 'M', 'Age': 22, 'Contact_Number': '8888800002', 'Email': 'ravi@student.com', 'Status': 'Active', 'Role_ID': 1, 'Image_Path': 'uploads/default.jpg', 'Emergency_Contact': '7777700002'}

  TABLE: Rooms  (1 records)
  {'Room_ID': 101, 'Room_Number': '101', 'Hostel_ID': 2, 'Floor_Number': 1, 'Capacity': 2}

  TABLE: Room Allocations  (2 records)
  {'Allocation_ID': 1, 'Member_ID': 110, 'Room_ID': 101, 'Allocat

In [11]:
rooms_table, _   = db_admin.get_table("stayease", "rooms")
members_table, _ = db_admin.get_table("stayease", "members")
_, room_final   = rooms_table.get(101)
_, member_final = members_table.get(112)

print("\n--- FINAL STATE AFTER BOTH THREADS ---")
print(f"  Room 101 Capacity   : {room_final['Capacity']}  (Expected: 2)")
print(f"  Member 112 Status   : {member_final['Status']}  (Expected: Active)")
print(f"  Thread 1 result     : {results.get('thread1')}  (Expected: committed)")
print(f"  Thread 2 result     : {results.get('thread2')}  (Expected: committed)")
print_all_tables()
print_wal_log()

isolation_ok = (
    room_final['Capacity'] == 2 and
    member_final['Status'] == 'Active' and
    results.get('thread1') == 'committed' and
    results.get('thread2') == 'committed'
)
print("\n" + ("ISOLATION TEST PASSED" if isolation_ok else "ISOLATION TEST FAILED"))


--- FINAL STATE AFTER BOTH THREADS ---
  Room 101 Capacity   : 2  (Expected: 2)
  Member 112 Status   : Active  (Expected: Active)
  Thread 1 result     : committed  (Expected: committed)
  Thread 2 result     : committed  (Expected: committed)

  TABLE: Members  (2 records)
  {'Member_ID': 110, 'First_Name': 'Priya', 'Last_Name': 'Sharma', 'Gender': 'F', 'Age': 20, 'Contact_Number': '8888800001', 'Email': 'priya@student.com', 'Status': 'Active', 'Role_ID': 1, 'Image_Path': 'uploads/default.jpg', 'Emergency_Contact': '7777700001'}
  {'Member_ID': 112, 'First_Name': 'Ravi', 'Last_Name': 'Kumar', 'Gender': 'M', 'Age': 22, 'Contact_Number': '8888800002', 'Email': 'ravi@student.com', 'Status': 'Active', 'Role_ID': 1, 'Image_Path': 'uploads/default.jpg', 'Emergency_Contact': '7777700002'}

  TABLE: Rooms  (1 records)
  {'Room_ID': 101, 'Room_Number': '101', 'Hostel_ID': 2, 'Floor_Number': 1, 'Capacity': 2}

  TABLE: Room Allocations  (2 records)
  {'Allocation_ID': 1, 'Member_ID': 110, 'Ro

---
## TEST 4: DURABILITY
### Once committed, data must survive a system restart (simulated by fresh DB instance).

In [12]:
section("TEST 4: DURABILITY — Committed Data Survives Restart")

print("\n--- PHASE 1: Committing data to disk ---")

db = DatabaseManager()
db.create_database("stayease")

db.create_table("stayease", "members", members_schema, search_key="Member_ID")
db.create_table("stayease", "rooms", rooms_schema, search_key="Room_ID")
db.create_table("stayease", "room_allocations", allocations_schema, search_key="Allocation_ID")

m, _ = db.get_table("stayease", "members")
r, _ = db.get_table("stayease", "rooms")
a, _ = db.get_table("stayease", "room_allocations")

db.begin()
m.insert({'Member_ID': 200, 'First_Name': 'Durability', 'Last_Name': 'Test',
    'Gender': 'M', 'Age': 25, 'Contact_Number': '9111111111',
    'Email': 'durability@test.com', 'Status': 'Active',
    'Role_ID': 1, 'Image_Path': 'uploads/default.png', 'Emergency_Contact': '9111111112'})
r.insert({'Room_ID': 400, 'Room_Number': '400', 'Hostel_ID': 2, 'Floor_Number': 4, 'Capacity': 2})
a.insert({'Allocation_ID': 50, 'Member_ID': 200, 'Room_ID': 400,
    'Allocation_Date': '2026-04-03', 'Check_Out_Date': 'None', 'Status': 'Active'})
db.commit("stayease")

print("  Committed: Member 200, Room 400, Allocation 50")
print("\n--- WAL LOG AFTER PHASE 1 COMMIT ---")
print_wal_log()

del db
print("\n  DB object deleted — simulating system shutdown...")


############################################################
  TEST 4: DURABILITY — Committed Data Survives Restart
############################################################

--- PHASE 1: Committing data to disk ---
BEGIN: Transaction started. [tx_id=4a13d832]
All tables in 'stayease' persisted to disk.
COMMIT: Changes saved.
LOCK RELEASED.
  Committed: Member 200, Room 400, Allocation 50

--- WAL LOG AFTER PHASE 1 COMMIT ---

  WAL LOG: stayease_wal.log
  >>> BEGIN    | tx_id=24fd21b7 | time=2026-04-04 22:25:45
  INSERT   | tx_id=24fd21b7 | time=2026-04-04 22:25:03 | table=members | key=110
  INSERT   | tx_id=24fd21b7 | time=2026-04-04 22:25:03 | table=members | key=112
  INSERT   | tx_id=24fd21b7 | time=2026-04-04 22:25:03 | table=rooms | key=101
  INSERT   | tx_id=24fd21b7 | time=2026-04-04 22:25:03 | table=room_allocations | key=1
  INSERT   | tx_id=24fd21b7 | time=2026-04-04 22:25:03 | table=room_allocations | key=2
  >>> COMMIT   | tx_id=24fd21b7 | time=2026-04-04 22:25:45
  

In [13]:
print("\n--- PHASE 2: Fresh instance — simulating restart ---")

db2 = DatabaseManager()
db2.create_database("stayease")
db2.create_table("stayease", "members", members_schema, search_key="Member_ID")
db2.create_table("stayease", "rooms", rooms_schema, search_key="Room_ID")
db2.create_table("stayease", "room_allocations", allocations_schema, search_key="Allocation_ID")

db2.reload_all("stayease")
db2.recover("stayease")

m2, _ = db2.get_table("stayease", "members")
r2, _ = db2.get_table("stayease", "rooms")
a2, _ = db2.get_table("stayease", "room_allocations")

member_found, member_data = m2.get(200)
room_found, room_data     = r2.get(400)
alloc_found, alloc_data   = a2.get(50)

print("\n--- VERIFICATION AFTER RESTART ---")
print(f"  Member 200 found    : {member_found}  (Expected: True)")
if member_found:
    print(f"  Name               : {member_data['First_Name']} {member_data['Last_Name']}")
print(f"  Room 400 found      : {room_found}  (Expected: True)")
if room_found:
    print(f"  Floor              : {room_data['Floor_Number']}")
print(f"  Allocation 50 found : {alloc_found}  (Expected: True)")
if alloc_found:
    print(f"  Status             : {alloc_data['Status']}")

print("\n--- WAL LOG AFTER RECOVERY ---")
print_wal_log()

durability_ok = member_found and room_found and alloc_found
print("\n" + ("DURABILITY TEST PASSED" if durability_ok else "DURABILITY TEST FAILED"))

print("\n--- CLEANUP ---")
db2.begin()
a2.delete(50)
r2.delete(400)
m2.delete(200)
db2.commit("stayease")
print("  Test records removed.")


--- PHASE 2: Fresh instance — simulating restart ---
Restored table 'members' from disk.
Restored table 'rooms' from disk.
Restored table 'room_allocations' from disk.
--- RECOVERY: Processing stayease_wal.log ---
RECOVERY: Replaying committed transaction...
  REDO INSERT: members key=110
  REDO INSERT: members key=112
  REDO INSERT: rooms key=101
  REDO INSERT: room_allocations key=1
  REDO INSERT: room_allocations key=2
RECOVERY: Replaying committed transaction...
RECOVERY: Replaying committed transaction...
  REDO DELETE: room_allocations key=50
  REDO DELETE: rooms key=400
  REDO DELETE: members key=200
RECOVERY: Replaying committed transaction...
  REDO INSERT: members key=130
  REDO INSERT: rooms key=303
  REDO INSERT: room_allocations key=20
RECOVERY: Replaying committed transaction...
  REDO DELETE: room_allocations key=20
  REDO DELETE: rooms key=303
  REDO DELETE: members key=130
RECOVERY: Replaying committed transaction...
  REDO UPDATE: rooms key=101
  REDO UPDATE: rooms k

---
## Final Summary

In [14]:
section("FINAL SUMMARY — ACID TEST RESULTS")
print(f"\n  Atomicity   : {'PASSED' if atomicity_ok else 'FAILED'}")
print(f"  Consistency : {'PASSED' if consistency_ok else 'FAILED'}")
print(f"  Isolation   : {'PASSED' if isolation_ok else 'FAILED'}")
print(f"  Durability  : {'PASSED' if durability_ok else 'FAILED'}")
print()
all_passed = atomicity_ok and consistency_ok and isolation_ok and durability_ok
print(f"  {'ALL ACID PROPERTIES VALIDATED SUCCESSFULLY' if all_passed else 'SOME TESTS FAILED — CHECK OUTPUT ABOVE'}")

print("\n--- FINAL DATABASE STATE ---")
print_all_tables()
print_wal_log()


############################################################
  FINAL SUMMARY — ACID TEST RESULTS
############################################################

  Atomicity   : PASSED
  Consistency : PASSED
  Isolation   : PASSED
  Durability  : PASSED

  ALL ACID PROPERTIES VALIDATED SUCCESSFULLY

--- FINAL DATABASE STATE ---

  TABLE: Members  (2 records)
  {'Member_ID': 110, 'First_Name': 'Priya', 'Last_Name': 'Sharma', 'Gender': 'F', 'Age': 20, 'Contact_Number': '8888800001', 'Email': 'priya@student.com', 'Status': 'Active', 'Role_ID': 1, 'Image_Path': 'uploads/default.jpg', 'Emergency_Contact': '7777700001'}
  {'Member_ID': 112, 'First_Name': 'Ravi', 'Last_Name': 'Kumar', 'Gender': 'M', 'Age': 22, 'Contact_Number': '8888800002', 'Email': 'ravi@student.com', 'Status': 'Active', 'Role_ID': 1, 'Image_Path': 'uploads/default.jpg', 'Emergency_Contact': '7777700002'}

  TABLE: Rooms  (1 records)
  {'Room_ID': 101, 'Room_Number': '101', 'Hostel_ID': 2, 'Floor_Number': 1, 'Capacity': 2}

 